# Attendance chatbot Phase 2/3: deterministic tests in Colab

This notebook runs deterministic Phase 2 runtime/evaluator tests and Phase 3 UI/acceptance tests from a sanitized source snapshot. It does not run live questions or the 231-case evaluation. For privacy, the ZIP's `tests.jsonl` is generated as a synthetic 231-row count-only manifest with placeholder questions and no original questions, employee names, employee IDs, or reference answers. It must not be used for evaluation. The ZIP also excludes `.env` files, credentials, attendance database rows, and prior result artifacts. The checks use mocked model/database boundaries; the T4 GPU is not needed for this notebook. Live model inference and PostgreSQL acceptance remain a separate step that must use only a synthetic test database, never a production attendance DSN.

## 1. Install the minimal test environment

This installs the online runtime's direct dependencies and test tools, rather than the repository's full ML environment.

In [ ]:
import subprocess
import sys

# Direct dependencies for deterministic mocked runtime, UI, and evaluator checks.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "pydantic>=2.11.9,<3",
        "python-dotenv>=1.1.1",
        "psycopg[binary]>=3.2.0",
        "openai>=1.109.1,<3",
        "litellm>=1.77.5",
        "pytest>=8,<10",
        "ruff>=0.14",
        "gradio>=5.47.2,<6.0",
        "chromadb>=1.0.20,<2",
        "langchain-huggingface>=1.0.0",
        "langchain-openai>=0.3.33",
        "sentence-transformers>=5.1.1",
        "tiktoken>=0.11.0",
        "sqlglot>=30.19.0,<31",
    ],
    check=True,
)
print("Minimal Colab test dependencies installed.")

## 2. Upload and validate the current source snapshot

Upload `attendance_phase2_source.zip` from the `colab` folder in this checkout, or pre-stage it at `/content/attendance_phase2_source.zip`. Extraction rejects path traversal, environment files, and result directories.


In [ ]:
from google.colab import files
from hashlib import sha256
from pathlib import Path, PurePosixPath
import shutil
from zipfile import ZipFile

archive_path = Path('/content/attendance_phase2_source.zip')
expected_archive_sha256 = "5ae8064aaa409aa9a064a6f56c34ab4467b8ccad65550cd312afc83c4d376e69"
if archive_path.is_file():
    archive_bytes = archive_path.read_bytes()
    if sha256(archive_bytes).hexdigest() != expected_archive_sha256:
        archive_path.unlink()
if not archive_path.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one attendance_phase2_source.zip file.')
    archive_name, archive_bytes = next(iter(uploaded.items()))
    if archive_name != 'attendance_phase2_source.zip':
        raise ValueError('The uploaded file must be named attendance_phase2_source.zip.')
    archive_path.write_bytes(archive_bytes)

if sha256(archive_bytes).hexdigest() != expected_archive_sha256:
    raise ValueError('Sanitized source archive SHA-256 mismatch.')
source_root = Path('/content/attendance_phase2_source')
if source_root.exists():
    expected_root = Path('/content/attendance_phase2_source').resolve()
    if source_root.resolve() != expected_root:
        raise ValueError('Source extraction path does not match the expected Colab directory.')
    shutil.rmtree(source_root)
source_root.mkdir(parents=True, exist_ok=True)
with ZipFile(archive_path) as bundle:
    members = bundle.infolist()
    names = [member.filename for member in members]
    for name in names:
        item = PurePosixPath(name)
        lowered = name.casefold()
        if item.is_absolute() or '..' in item.parts:
            raise ValueError(f'Unsafe archive member: {name}')
        if item.name.casefold().startswith('.env') or 'credential' in lowered:
            raise ValueError(f'Secret configuration must not be in the source ZIP: {name}')
        if '/results/' in f'/{lowered}/' or lowered.endswith('.tmp'):
            raise ValueError(f'Result artifacts must not be in the source ZIP: {name}')
    required = {
        'week5/new_app.py',
        'week5/test_new_app.py',
        'week5/new_implementation/chroma_client.py',
        'week5/new_implementation/config.py',
        'week5/new_implementation/embedding.py',
        'week5/new_implementation/test_config.py',
        'week5/new_implementation/colab/package_source.py',
        'week5/new_implementation/colab/private_payload.py',
        'week5/new_implementation/colab/test_private_payload.py',
        'week5/new_implementation/colab/prepare_synthetic_runtime.py',
        'week5/new_implementation/colab/test_prepare_synthetic_runtime.py',
        'week5/new_implementation/colab/prepare_private_runtime.py',
        'week5/new_implementation/colab/run_private_eval.py',
        'week5/new_implementation/colab/cleanup_private_runtime.py',
        'week5/new_implementation/colab/test_private_runtime.py',
        'week5/new_implementation/colab/run_acceptance_turn.py',
        'week5/new_implementation/colab/run_synthetic_eval.py',
        'week5/new_implementation/colab/synthetic_evaluation_cases.jsonl',
        'week5/new_implementation/online/pipeline.py',
        'week5/new_implementation/online/context.py',
        'week5/new_implementation/online/running_total.py',
        'week5/new_implementation/tests/online/test_pipeline.py',
        'week5/new_evaluation/acceptance.py',
        'week5/new_evaluation/test_eval.py',
        'week5/new_evaluation/test_acceptance.py',
        'week5/new_evaluation/tests.jsonl',
    }
    missing = required.difference(names)
    if missing:
        raise ValueError(f'Source ZIP is missing required files: {sorted(missing)}')
    bundle.extractall(source_root)
repo_root = source_root
print(f'Validated {len(names)} source files under {repo_root}; SHA-256 verified.')

## 3. Run deterministic Phase 2 and Phase 3 checks

These tests use mocked model/database boundaries. They make no LLM calls and do not connect to PostgreSQL. They include the Gradio callback/session tests, evaluator checks, and one-turn acceptance checkpoint tests. The corpus-size unit test loads the generated synthetic 231-row manifest only; it does not load or score the private evaluation corpus.

In [ ]:
import os
import subprocess
import sys

os.chdir(repo_root)
test_paths = [
    "week5/new_implementation/tests/online",
    "week5/new_implementation/test_config.py",
    "week5/new_implementation/colab/test_prepare_synthetic_runtime.py",
    "week5/new_implementation/colab/test_private_payload.py",
    "week5/new_implementation/colab/test_private_runtime.py",
    "week5/new_evaluation/test_eval.py",
    "week5/new_evaluation/test_acceptance.py",
    "week5/test_new_app.py",
]
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", *test_paths],
    text=True,
    capture_output=True,
)
if result.stdout:
    print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
print(f"pytest exit code: {result.returncode}")
if result.returncode:
    raise SystemExit(result.returncode)

## 4. Check formatting, lint, and compilation

These checks cover the online runtime, evaluator, acceptance runner, and Gradio callbacks/tests. Ruff uses isolated rules E4, E7, E9, and F so Colab-level settings cannot change the lint result.

In [ ]:
checks = [
    [sys.executable, "-m", "ruff", "check", "--isolated", "--select", "E4,E7,E9,F",
     "week5/new_implementation/online",
     "week5/new_implementation/test_config.py",
     "week5/new_implementation/colab/package_source.py",
     "week5/new_implementation/colab/private_payload.py",
     "week5/new_implementation/colab/test_private_payload.py",
     "week5/new_implementation/colab/prepare_synthetic_runtime.py",
     "week5/new_implementation/colab/test_prepare_synthetic_runtime.py",
     "week5/new_implementation/colab/prepare_private_runtime.py",
     "week5/new_implementation/colab/run_private_eval.py",
     "week5/new_implementation/colab/cleanup_private_runtime.py",
     "week5/new_implementation/colab/test_private_runtime.py",
     "week5/new_implementation/colab/run_acceptance_turn.py",
     "week5/new_implementation/colab/run_synthetic_eval.py",
     "week5/new_evaluation/eval.py",
     "week5/new_evaluation/test_eval.py",
     "week5/new_evaluation/acceptance.py",
     "week5/new_evaluation/test_acceptance.py",
     "week5/test_new_app.py",
     "week5/new_app.py"],
    [sys.executable, "-m", "ruff", "format", "--check", "--isolated",
     "week5/new_implementation/online",
     "week5/new_implementation/test_config.py",
     "week5/new_implementation/colab/package_source.py",
     "week5/new_implementation/colab/private_payload.py",
     "week5/new_implementation/colab/test_private_payload.py",
     "week5/new_implementation/colab/prepare_synthetic_runtime.py",
     "week5/new_implementation/colab/test_prepare_synthetic_runtime.py",
     "week5/new_implementation/colab/prepare_private_runtime.py",
     "week5/new_implementation/colab/run_private_eval.py",
     "week5/new_implementation/colab/cleanup_private_runtime.py",
     "week5/new_implementation/colab/test_private_runtime.py",
     "week5/new_implementation/colab/run_acceptance_turn.py",
     "week5/new_implementation/colab/run_synthetic_eval.py",
     "week5/new_evaluation/eval.py",
     "week5/new_evaluation/test_eval.py",
     "week5/new_evaluation/acceptance.py",
     "week5/new_evaluation/test_acceptance.py",
     "week5/test_new_app.py",
     "week5/new_app.py"],
    [sys.executable, "-m", "compileall", "-q",
     "week5/new_implementation/online",
     "week5/new_implementation/test_config.py",
     "week5/new_implementation/colab/package_source.py",
     "week5/new_implementation/colab/private_payload.py",
     "week5/new_implementation/colab/prepare_synthetic_runtime.py",
     "week5/new_implementation/colab/test_prepare_synthetic_runtime.py",
     "week5/new_implementation/colab/prepare_private_runtime.py",
     "week5/new_implementation/colab/run_private_eval.py",
     "week5/new_implementation/colab/cleanup_private_runtime.py",
     "week5/new_implementation/colab/run_acceptance_turn.py",
     "week5/new_implementation/colab/run_synthetic_eval.py",
     "week5/new_evaluation/eval.py",
     "week5/new_evaluation/acceptance.py",
     "week5/new_app.py"],
]
for command in checks:
    print("$", " ".join(command))
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)
    if result.returncode:
        raise SystemExit(result.returncode)
print("Lint, formatting, and compilation checks passed.")

## Live model/database checks are separate

This notebook intentionally does not configure Ollama or connect to PostgreSQL. A live check needs a Colab GPU runtime and a local **synthetic test database** containing no real attendance rows. Do not expose a production database through a tunnel. Keep any later test DSN in Colab Secrets; never add it to this notebook or ZIP. Run named acceptance scenarios one turn at a time with a checkpoint; inspect each outcome before running the next turn.